In [187]:
import matplotlib.pyplot as plt
import nelpy as nel
import nelpy.plotting as npl
import numpy as np
import seaborn as sns
from neuro_py.io import loading
from neuro_py.plotting.figure_helpers import set_size
from neuro_py.tuning import maps
from skimage import measure
import os
import pandas as pd
import scipy.io as sio
import logging

logging.getLogger().setLevel(logging.ERROR)
basepath =  r"Z:\Data\AYAold\AYA10\day34"
basename=os.path.basename(basepath)# Define the new folder path
#create working folder
new_folder_path = os.path.join(basepath, "complex_system")
brain_region="CA1"
animal="mouse"   #or Mouse
cell_type='all'
path_parts = basepath.split("\\")  # Split the path using backslash
animal_name = path_parts[-2]  # Second-to-last part is the animal name
day = path_parts[-1]  # Last part is the day
save_dir = r"D:\Documents\GitHub\R_Project\PPP\codebook\Criticality_analysis\complex_system\manifold"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
# Create the folder if it doesn't exist
if not os.path.exists(new_folder_path):
    os.makedirs(new_folder_path)
    print(f"Folder created: {new_folder_path}")
else:
    print(f"Folder already exists: {new_folder_path}")
# load in spike data from hpc pyramidal cells

# load theta epochs
state_dict = loading.load_SleepState_states(basepath)
nrem_epochs = nel.EpochArray(
    state_dict["NREMstate"],
)
awake_epochs = nel.EpochArray(
    state_dict["WAKEstate"],
)
# theta_epochs = nel.EpochArray(
#     state_dict["THETA"],
# )
rem_epochs=nel.EpochArray(state_dict["REMstate"],)
try:
    # Try default loader
    epoch_df = loading.load_epoch(basepath)
except Exception as e:
    print(f"load_epoch failed with error: {e}")
    print("Falling back to manual .mat file load...")

    # Load the .mat file
    mat = sio.loadmat(os.path.join(basepath, "DayNight25.MergePoints.events.mat"), squeeze_me=True)

    # Extract the 'MergePoints' structure
    merge_points = mat['MergePoints']

    # Extract the timestamps array: shape (N, 2)
    timestamps = merge_points['timestamps']  # this is the field you're interested in
    # Unwrap the array inside
    timestamps = np.array(timestamps.tolist())

    # Now you should have shape (N, 2)
    epoch_df = pd.DataFrame(timestamps, columns=["startTime", "stopTime"])

sleep_epochs = nel.EpochArray(np.array([epoch_df.startTime, epoch_df.stopTime]).T)

# Load CA1 spikes
st_ca1, cm_ca1 = loading.load_spikes(basepath, brainRegion="CA1")
E_spikes_ca1 = st_ca1.time[(cm_ca1.putativeCellType == 'Pyramidal Cell')]
I_spikes_ca1 = st_ca1.time[(cm_ca1.putativeCellType.str.contains("Interneuron"))]
n_E_ca1 = len(E_spikes_ca1)
n_I_ca1 = len(I_spikes_ca1)
n_total_ca1 = len(st_ca1.time)
print(f"CA1 - Total cells: {n_total_ca1}, Excitatory: {n_E_ca1}, Interneurons: {n_I_ca1}")

# Load CA2 spikes
st_ca2, cm_ca2 = loading.load_spikes(basepath, brainRegion="CA2")
E_spikes_ca2 = st_ca2.time[(cm_ca2.putativeCellType == 'Pyramidal Cell')]
I_spikes_ca2 = st_ca2.time[(cm_ca2.putativeCellType.str.contains("Interneuron"))]
n_E_ca2 = len(E_spikes_ca2)
n_I_ca2 = len(I_spikes_ca2)
n_total_ca2 = len(st_ca2.time)
print(f"CA2 - Total cells: {n_total_ca2}, Excitatory: {n_E_ca2}, Interneurons: {n_I_ca2}")

# Load CA3 spikes
st_ca3, cm_ca3 = loading.load_spikes(basepath, brainRegion="CA3")
E_spikes_ca3 = st_ca3.time[(cm_ca3.putativeCellType == 'Pyramidal Cell')]
I_spikes_ca3 = st_ca3.time[(cm_ca3.putativeCellType.str.contains("Interneuron"))]
n_E_ca3 = len(E_spikes_ca3)
n_I_ca3 = len(I_spikes_ca3)
n_total_ca3 = len(st_ca3.time)
print(f"CA3 - Total cells: {n_total_ca3}, Excitatory: {n_E_ca3}, Interneurons: {n_I_ca3}")

Folder already exists: Z:\Data\AYAold\AYA10\day34\complex_system
CA1 - Total cells: 24, Excitatory: 21, Interneurons: 3
CA2 - Total cells: 20, Excitatory: 15, Interneurons: 5
CA3 - Total cells: 35, Excitatory: 33, Interneurons: 2


In [189]:
import os, re, numpy as np, pandas as pd
import glob  # <-- add this
def _load_mat_any(mat_path):
    from scipy.io import loadmat
    try:
        return loadmat(
            mat_path,
            squeeze_me=True,
            struct_as_record=False,
            simplify_cells=True
        )
    except TypeError:
        return loadmat(
            mat_path,
            squeeze_me=True,
            struct_as_record=False
        )
    except NotImplementedError:
        import mat73
        return mat73.loadmat(mat_path)
def _to_1d_array(x):
    """
    Robustly coerce MATLAB-loaded field to a 1D numpy array.
    - Scalars -> 1-length array
    - Lists/tuples/ndarrays -> flattened 1D
    - None / missing -> empty array
    """
    if x is None:
        return np.array([], dtype=float)
    arr = np.asarray(x, dtype=object)  # start as object to avoid surprises
    if arr.ndim == 0:
        # scalar or unsized object
        return np.atleast_1d(arr).astype(float, copy=False)
    # flatten and try float cast (ignore non-numeric gracefully)
    try:
        return np.asarray(arr).ravel().astype(float, copy=False)
    except (ValueError, TypeError):
        # fallback: best-effort, convert elementwise
        return np.array([float(v) for v in np.ravel(arr)], dtype=float)

def _to_state_str(s):
    # MATLAB char arrays sometimes come in as numpy arrays
    try:
        if isinstance(s, (bytes, bytearray)):
            return s.decode('utf-8', 'ignore')
        if isinstance(s, str):
            return s
        # arrays or weird containers -> join characters
        arr = np.asarray(s)
        if arr.ndim == 0:
            return str(arr.item())
        # e.g., array of characters
        return ''.join(arr.tolist()) if arr.dtype.kind in ('U','S') else str(arr)
    except Exception:
        return str(s)

def load_d2_timeline_complete(basepath, region):
    """Complete version that loads ALL sleep-state data, including d2_err."""
    region = region.upper()
    search_dir = os.path.join(basepath, "complex_system")
    # pats = [f"*_{region}_D2_time_resolved_rippleablate.mat", f"*_{region}_D2_time_rippleablate.mat"]
    pats = [f"*_{region}_D2_time_resolved_error_bar.mat", f"*_{region}_D2_time_error_bar.mat"]
    files = []
    for pat in pats:
        files.extend(glob.glob(os.path.join(search_dir, pat)))
    if not files:
        raise FileNotFoundError(f"No *_{region}_D2_time*.mat under {search_dir}")

    mat_path = max(files, key=os.path.getmtime)
    print(f"Loading: {os.path.basename(mat_path)}")

    mat = _load_mat_any(mat_path)
    results = mat['results']
    params = mat.get('params', {})

    # Make results iterable no matter if it's a list, ndarray, or dict
    if isinstance(results, dict):
        iter_results = list(results.values())
    else:
        iter_results = list(results if isinstance(results, (list, tuple)) else np.atleast_1d(results))

    print(f"📊 Found {len(iter_results)} result items (sleep states)")
    print(f"📊 Parameters: {params}")

    # Try to infer animal/day from filename if available
    fname = os.path.basename(mat_path)
    animal_name = None
    day = None
    m_day = re.search(r'day(\d+)', fname, flags=re.I)
    if m_day:
        day = int(m_day.group(1))
    m_animal = re.match(r'([^_]+)_day\d+_', fname)
    if m_animal:
        animal_name = m_animal.group(1)

    all_centers, all_d2, all_d2err, all_states = [], [], [], []
    n_states_with_err = 0

    for i, result_item in enumerate(iter_results):
        # Access as dict-like regardless of MATLAB struct quirks
        state = _to_state_str(result_item.get('state', 'UNKNOWN')) if hasattr(result_item, 'get') else \
                _to_state_str(result_item['state']) if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) else 'UNKNOWN'

        centers_raw = result_item.get('center', []) if hasattr(result_item, 'get') else \
                      (result_item['center'] if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) and 'center' in result_item else [])
        d2_raw = result_item.get('d2', []) if hasattr(result_item, 'get') else \
                 (result_item['d2'] if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) and 'd2' in result_item else [])
        d2err_raw = result_item.get('d2_err', []) if hasattr(result_item, 'get') else \
                    (result_item['d2_err'] if (isinstance(result_item, dict) or hasattr(result_item, '__getitem__')) and 'd2_err' in result_item else [])

        centers  = _to_1d_array(centers_raw)
        d2_vals  = _to_1d_array(d2_raw)
        d2_err   = _to_1d_array(d2err_raw)

        # Presence flag (some states may not have d2_err)
        has_err = d2_err.size > 0 and np.any(np.isfinite(d2_err))
        n_states_with_err += int(has_err)

        # Harmonize lengths: truncate all to the common min
        sizes = [centers.size, d2_vals.size] + ([d2_err.size] if has_err else [])
        n_pts = min(sizes) if min(sizes) > 0 else 0
        print(f"   State {i}: {state} - requested points={centers.size}, actual used={n_pts}")

        if n_pts == 0:
            print("     ⚠️ Empty state - no data points")
            continue

        centers = centers[:n_pts]
        d2_vals = d2_vals[:n_pts]
        d2_err  = (d2_err[:n_pts] if has_err else np.full(n_pts, np.nan, dtype=float))

        # Ranges
        try:
            print(f"     Time range: {float(np.nanmin(centers))} to {float(np.nanmax(centers))}")
        except ValueError:
            print("     Time range: (all NaN)")

        try:
            print(f"     D2 range: {float(np.nanmin(d2_vals)):.4f} to {float(np.nanmax(d2_vals)):.4f}")
        except ValueError:
            print("     D2 range: (all NaN)")

        if np.any(np.isfinite(d2_err)):
            try:
                dmin, dmax = float(np.nanmin(d2_err)), float(np.nanmax(d2_err))
                print(f"     D2_ERR range: {dmin:.4f} to {dmax:.4f}")
            except ValueError:
                print("     D2_ERR range: (all NaN)")
        else:
            print("     D2_ERR: missing → filled with NaN")

        all_centers.append(centers)
        all_d2.append(d2_vals)
        all_d2err.append(d2_err)
        all_states.append(np.array([state]*n_pts, dtype=object))

    if len(all_centers) == 0:
        raise ValueError("No data found in any sleep state - all states are empty")

    # Concatenate
    all_centers = np.concatenate(all_centers)
    all_d2      = np.concatenate(all_d2)
    all_d2err   = np.concatenate(all_d2err)
    all_states  = np.concatenate(all_states)

    # DataFrame (include d2_err)
    df = pd.DataFrame({
        'center_s': all_centers.astype(float, copy=False),
        'd2':       all_d2.astype(float, copy=False),
        'd2_err':   all_d2err.astype(float, copy=False),
        'state':    all_states.astype(str)
    }).sort_values('center_s').reset_index(drop=True)

    print(f"\n✅ Complete timeline created:")
    print(f"📊 Total data points: {len(df)}")
    print(f"📊 Time range: {df['center_s'].min()} to {df['center_s'].max()}")
    print(f"📊 Duration: {(df['center_s'].max() - df['center_s'].min()):.1f} seconds")
    print(f"📊 State distribution:")
    print(df['state'].value_counts())

    # Empty-state tracker
    empty_states = []
    for ri in iter_results:
        st = _to_state_str(ri.get('state','UNKNOWN')) if hasattr(ri,'get') else \
             _to_state_str(ri['state']) if (isinstance(ri,dict) or hasattr(ri,'__getitem__')) and 'state' in ri else 'UNKNOWN'
        csz = np.size(np.atleast_1d(ri.get('center', []))) if hasattr(ri,'get') else \
              np.size(np.atleast_1d(ri['center'])) if (isinstance(ri,dict) or hasattr(ri,'__getitem__')) and 'center' in ri else 0
        if csz == 0:
            empty_states.append(st)

    meta = {
        "animal_name": animal_name,
        "day": day,
        "region": region,
        "path": mat_path,
        "params": params,
        "columns": list(df.columns),
        "shape": df.shape,
        "n_states": len(iter_results),
        "states_with_d2_err": n_states_with_err,
        "empty_states": empty_states
    }

    return df, meta



# Test the complete loading function
try:
    ca1_complete_df, ca1_complete_meta = load_d2_timeline_complete(basepath, "CA1")
    
    # pfc_complete_df, pfc_complete_meta = load_d2_timeline_complete(basepath, "PFC")

    
except Exception as e:
    print(f"❌ Failed: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "="*60)

Loading: day34_CA1_D2_time_resolved_error_bar.mat
📊 Found 3 result items (sleep states)
📊 Parameters: {'window_size': 30, 'step_size': 2, 'guard_sec': 5, 'deltaT': 0.04, 'order': 20, 'quant': 0.1, 'brain_region': 'CA1', 'n_cells': 31, 'total_spikes': 238125}
   State 0: NREM - requested points=2241, actual used=2241
     Time range: 277.0 to 13315.0
     D2 range: 0.0452 to 0.2904
     D2_ERR range: 0.0168 to 0.0425
   State 1: REM - requested points=175, actual used=175
     Time range: 3218.0 to 13113.0
     D2 range: 0.0285 to 0.2000
     D2_ERR range: 0.0127 to 0.0342
   State 2: WAKE - requested points=3409, actual used=3409
     Time range: 21.0 to 14258.0
     D2 range: 0.0250 to 0.3182
     D2_ERR range: 0.0109 to 0.0447

✅ Complete timeline created:
📊 Total data points: 5825
📊 Time range: 21.0 to 14258.0
📊 Duration: 14237.0 seconds
📊 State distribution:
state
WAKE    3409
NREM    2241
REM      175
Name: count, dtype: int64



In [190]:
import numpy as np
from scipy.signal import welch
import nelpy as nel

def _flatten_spike_times(spike_times):
    """Accepts list/tuple/array of arrays or a single 1D array, returns sorted 1D float array."""
    if isinstance(spike_times, (list, tuple)):
        # list of arrays or scalars
        if len(spike_times) == 0:
            return np.array([], dtype=float)
        if hasattr(spike_times[0], "__len__") and not np.isscalar(spike_times[0]):
            valid_arrays = [np.asarray(st, dtype=float).ravel()
                           for st in spike_times if st is not None and len(st) > 0]
            if len(valid_arrays) == 0:
                return np.array([], dtype=float)
            times = np.concatenate(valid_arrays)
        else:
            times = np.asarray(spike_times, dtype=float).ravel()
    else:
        arr = np.asarray(spike_times, dtype=object)
        if arr.dtype == object:
            valid_arrays = [np.asarray(st, dtype=float).ravel()
                           for st in arr if st is not None and len(st) > 0]
            if len(valid_arrays) == 0:
                return np.array([], dtype=float)
            times = np.concatenate(valid_arrays)
        else:
            times = arr.astype(float).ravel()

    times = times[np.isfinite(times)]
    if times.size == 0:
        return np.array([], dtype=float)  # Return empty array instead of raising error
    return np.sort(times)

def bin_spikes(spike_times, bin_size=0.04, t_start=None, t_stop=None):
    st = _flatten_spike_times(spike_times)
    
    # Handle empty spike times
    if len(st) == 0:
        # Return empty arrays
        return np.array([]), np.array([])
    
    if t_start is None: t_start = float(np.floor(st.min()))
    if t_stop  is None: t_stop  = float(np.ceil(st.max()))
    edges = np.arange(t_start, t_stop + bin_size, bin_size)
    counts, _ = np.histogram(st, bins=edges)
    rates = counts / bin_size
    t = edges[:-1] + bin_size/2
    return t, rates

# def segments_from_epochs(t, x, epochs):
#     arr = epochs.data if isinstance(epochs, nel.EpochArray) else np.asarray(epochs)
#     return [x[(t >= s) & (t < e)] for s, e in arr if np.any((t >= s) & (t < e))]


In [191]:
# Create plot for random 200 second segment containing all states
import matplotlib.pyplot as plt
import numpy as np
import random

# First compute MUA from spike times
print("Computing MUA firing rate from spike times...")
print("="*50)

# Bin spikes for CA1 if not empty
if n_total_ca1 > 0:
    print("\nCA1 spike binning:")
    time_bins_ca1, firing_rate_ca1 = bin_spikes(st_ca1.time, bin_size=0.04)
    
    if n_E_ca1 > 0:
        time_bins_ca1_E, firing_rate_ca1_E = bin_spikes(E_spikes_ca1, bin_size=0.04)
        print(f"  CA1 Excitatory binned - rate range: {firing_rate_ca1_E.min():.2f} to {firing_rate_ca1_E.max():.2f} Hz")
    else:
        print("  CA1 Excitatory: No spikes")
        
    if n_I_ca1 > 0:
        time_bins_ca1_I, firing_rate_ca1_I = bin_spikes(I_spikes_ca1, bin_size=0.04)
        print(f"  CA1 Interneurons binned - rate range: {firing_rate_ca1_I.min():.2f} to {firing_rate_ca1_I.max():.2f} Hz")
    else:
        print("  CA1 Interneurons: No spikes")
else:
    print("\nCA1: No cells found")

# Bin spikes for CA2 if not empty
if n_total_ca2 > 0:
    print("\nCA2 spike binning:")
    time_bins_ca2, firing_rate_ca2 = bin_spikes(st_ca2.time, bin_size=0.04)
    
    if n_E_ca2 > 0:
        time_bins_ca2_E, firing_rate_ca2_E = bin_spikes(E_spikes_ca2, bin_size=0.04)
        print(f"  CA2 Excitatory binned - rate range: {firing_rate_ca2_E.min():.2f} to {firing_rate_ca2_E.max():.2f} Hz")
    else:
        print("  CA2 Excitatory: No spikes")
        
    if n_I_ca2 > 0:
        time_bins_ca2_I, firing_rate_ca2_I = bin_spikes(I_spikes_ca2, bin_size=0.04)
        print(f"  CA2 Interneurons binned - rate range: {firing_rate_ca2_I.min():.2f} to {firing_rate_ca2_I.max():.2f} Hz")
    else:
        print("  CA2 Interneurons: No spikes")
else:
    print("\nCA2: No cells found")

# Bin spikes for CA3 if not empty
if n_total_ca3 > 0:
    print("\nCA3 spike binning:")
    time_bins_ca3, firing_rate_ca3 = bin_spikes(st_ca3.time, bin_size=0.04)
    
    if n_E_ca3 > 0:
        time_bins_ca3_E, firing_rate_ca3_E = bin_spikes(E_spikes_ca3, bin_size=0.04)
        print(f"  CA3 Excitatory binned - rate range: {firing_rate_ca3_E.min():.2f} to {firing_rate_ca3_E.max():.2f} Hz")
    else:
        print("  CA3 Excitatory: No spikes")
        
    if n_I_ca3 > 0:
        time_bins_ca3_I, firing_rate_ca3_I = bin_spikes(I_spikes_ca3, bin_size=0.04)
        print(f"  CA3 Interneurons binned - rate range: {firing_rate_ca3_I.min():.2f} to {firing_rate_ca3_I.max():.2f} Hz")
    else:
        print("  CA3 Interneurons: No spikes")
else:
    print("\nCA3: No cells found")

Computing MUA firing rate from spike times...

CA1 spike binning:
  CA1 Excitatory binned - rate range: 0.00 to 425.00 Hz
  CA1 Interneurons binned - rate range: 0.00 to 100.00 Hz

CA2 spike binning:
  CA2 Excitatory binned - rate range: 0.00 to 300.00 Hz
  CA2 Interneurons binned - rate range: 0.00 to 400.00 Hz

CA3 spike binning:
  CA3 Excitatory binned - rate range: 0.00 to 375.00 Hz
  CA3 Interneurons binned - rate range: 0.00 to 350.00 Hz


In [192]:
import numpy as np
import pandas as pd

# --- Inputs assumed to exist ---
# time_bins_ca1, firing_rate_ca1, time_bins_ca1_E, firing_rate_ca1_E, time_bins_ca1_I, firing_rate_ca1_I
# time_bins_ca2, firing_rate_ca2, time_bins_ca2_E, firing_rate_ca2_E, time_bins_ca2_I, firing_rate_ca2_I
# time_bins_ca3, firing_rate_ca3, time_bins_ca3_E, firing_rate_ca3_E, time_bins_ca3_I, firing_rate_ca3_I
# ca1_complete_df          # has a 'center_s' column

def compute_mean_firing_rate_per_unit(time_bins, firing_rate, n_units, df, region_name, cell_type_name):
    """Compute mean firing rate per unit and add to dataframe"""
    if n_units == 0:
        print(f"{region_name} {cell_type_name}: No units - skipping")
        return

    # 1) Make sure we have bin centers
    tb = np.asarray(time_bins, float).ravel()
    fr = np.asarray(firing_rate, float).ravel()

    # Handle empty spike arrays (0 cells with spikes)
    if tb.size == 0 or fr.size == 0:
        print(f"{region_name} {cell_type_name}: No spike data - filling with NaN")
        col_name = f"{region_name}_{cell_type_name}_FR_per_unit"
        df[col_name] = np.nan
        return

    if tb.size == fr.size + 1:  # edges → make centers
        t = 0.5 * (tb[:-1] + tb[1:])
    elif tb.size == fr.size:  # already centers
        t = tb
    else:
        raise ValueError("time_bins length must equal firing_rate length or be exactly one longer (edges).")

    # 2) Clean + sort
    ok = np.isfinite(t) & np.isfinite(fr)
    t, fr = t[ok], fr[ok]

    # If no valid data after filtering, fill with NaN
    if len(t) == 0:
        print(f"{region_name} {cell_type_name}: No valid data after filtering - filling with NaN")
        col_name = f"{region_name}_{cell_type_name}_FR_per_unit"
        df[col_name] = np.nan
        return

    ordr = np.argsort(t)
    t, fr = t[ordr], fr[ordr]

    # 3) Compute mean firing rate per unit (Hz per unit)
    fr_per_unit = fr / n_units

    # 4) 30 s centered moving average
    # Handle case where we have too few points for meaningful window
    if len(t) < 2:
        # Not enough points to compute diff - use default bin size
        dt = 0.04
    else:
        dt = float(np.median(np.diff(t)))
        # Check for NaN (can happen if all diffs are NaN)
        if not np.isfinite(dt) or dt <= 0:
            dt = 0.04

    win_s = 30.0
    win_n = max(int(round(win_s / dt)), 1)

    fr_series = pd.Series(fr_per_unit, index=pd.Index(t, name="t"))
    fr_30s = fr_series.rolling(window=win_n, center=True, min_periods=max(1, win_n//4)).mean().to_numpy()

    # 5) Interpolate onto D² centers
    centers = np.asarray(df["center_s"], float)
    fr_at_centers = np.interp(centers, t, fr_30s, left=np.nan, right=np.nan)

    # 6) Save to dataframe (Hz per unit)
    col_name = f"{region_name}_{cell_type_name}_FR_per_unit"
    df[col_name] = fr_at_centers
    print(f"{region_name} {cell_type_name}: Added column '{col_name}' (mean: {np.nanmean(fr_at_centers):.3f} Hz/unit)")

# Process CA1
if n_total_ca1 > 0:
    print("\n" + "="*50)
    print("Processing CA1 mean firing rates per unit:")
    print("="*50)
    compute_mean_firing_rate_per_unit(time_bins_ca1, firing_rate_ca1, n_total_ca1, 
                                      ca1_complete_df, "CA1", "All")
    if n_E_ca1 > 0:
        compute_mean_firing_rate_per_unit(time_bins_ca1_E, firing_rate_ca1_E, n_E_ca1, 
                                          ca1_complete_df, "CA1", "E")
    if n_I_ca1 > 0:
        compute_mean_firing_rate_per_unit(time_bins_ca1_I, firing_rate_ca1_I, n_I_ca1, 
                                          ca1_complete_df, "CA1", "I")

# Process CA2
if n_total_ca2 > 0:
    print("\n" + "="*50)
    print("Processing CA2 mean firing rates per unit:")
    print("="*50)
    compute_mean_firing_rate_per_unit(time_bins_ca2, firing_rate_ca2, n_total_ca2, 
                                      ca1_complete_df, "CA2", "All")
    if n_E_ca2 > 0:
        compute_mean_firing_rate_per_unit(time_bins_ca2_E, firing_rate_ca2_E, n_E_ca2, 
                                          ca1_complete_df, "CA2", "E")
    if n_I_ca2 > 0:
        compute_mean_firing_rate_per_unit(time_bins_ca2_I, firing_rate_ca2_I, n_I_ca2, 
                                          ca1_complete_df, "CA2", "I")

# Process CA3
if n_total_ca3 > 0:
    print("\n" + "="*50)
    print("Processing CA3 mean firing rates per unit:")
    print("="*50)
    compute_mean_firing_rate_per_unit(time_bins_ca3, firing_rate_ca3, n_total_ca3, 
                                      ca1_complete_df, "CA3", "All")
    if n_E_ca3 > 0:
        compute_mean_firing_rate_per_unit(time_bins_ca3_E, firing_rate_ca3_E, n_E_ca3, 
                                          ca1_complete_df, "CA3", "E")
    if n_I_ca3 > 0:
        compute_mean_firing_rate_per_unit(time_bins_ca3_I, firing_rate_ca3_I, n_I_ca3, 
                                          ca1_complete_df, "CA3", "I")

print("\n" + "="*50)
print("Summary of added columns:")
print(ca1_complete_df.filter(regex='FR_per_unit').columns.tolist())
print("="*50)


Processing CA1 mean firing rates per unit:
CA1 All: Added column 'CA1_All_FR_per_unit' (mean: 0.705 Hz/unit)
CA1 E: Added column 'CA1_E_FR_per_unit' (mean: 0.674 Hz/unit)
CA1 I: Added column 'CA1_I_FR_per_unit' (mean: 0.916 Hz/unit)

Processing CA2 mean firing rates per unit:
CA2 All: Added column 'CA2_All_FR_per_unit' (mean: 2.426 Hz/unit)
CA2 E: Added column 'CA2_E_FR_per_unit' (mean: 1.015 Hz/unit)
CA2 I: Added column 'CA2_I_FR_per_unit' (mean: 6.657 Hz/unit)

Processing CA3 mean firing rates per unit:
CA3 All: Added column 'CA3_All_FR_per_unit' (mean: 2.185 Hz/unit)
CA3 E: Added column 'CA3_E_FR_per_unit' (mean: 0.613 Hz/unit)
CA3 I: Added column 'CA3_I_FR_per_unit' (mean: 28.115 Hz/unit)

Summary of added columns:
['CA1_All_FR_per_unit', 'CA1_E_FR_per_unit', 'CA1_I_FR_per_unit', 'CA2_All_FR_per_unit', 'CA2_E_FR_per_unit', 'CA2_I_FR_per_unit', 'CA3_All_FR_per_unit', 'CA3_E_FR_per_unit', 'CA3_I_FR_per_unit']


SWR and barrage

In [194]:
import numpy as np
import scipy.io as sio      
import scipy
# Load ripple events
ripple_events = loading.load_ripples_events(basepath=basepath)
ripple_epochs = nel.EpochArray(np.array([ripple_events.start, ripple_events.stop]).T)
ripple_peaks=ripple_events.peaks
# Load barrage events
ripple_epochs=nrem_epochs&ripple_epochs
try:
    # Load barrage events
    #brpath=r"W:\data\PPP\CA2\PPP7\day8"
    # barrage_events = loading.load_barrage_events(basepath=basepath)  
    # # Create barrage epochs
    # barrage_epochs = nel.EpochArray(np.array([barrage_events.start, barrage_events.stop]).T)
    # # Find barrages in NREM epochs
    # barrages_in_nrem = barrage_epochs & nrem_epochs
    mat_path = os.path.join(basepath, r"BARR_archive", f'{basename}.HSEn2.events.mat')
    #mat_path = os.path.join(basepath, f'{basename}.HSEn2.events.mat')

    data = scipy.io.loadmat(mat_path, struct_as_record=False, squeeze_me=True)
    barrage = data['HSEn2']
    barrage_timestamps = barrage.timestamps 
    
    # Create barrage epochs from timestamps
    if barrage_timestamps.ndim == 1:
        # Single barrage or timestamps need to be reshaped
        barrage_timestamps = barrage_timestamps.reshape(-1, 2)
    barrage_epochs = nel.EpochArray(barrage_timestamps)
    barrage_epochs = nrem_epochs & barrage_epochs
    
    # Compute barrage rate for each center_s with 30s window
    print("\n" + "="*50)
    print("Computing barrage rate with 30s window:")
    print("="*50)
    
    centers = ca1_complete_df["center_s"].values
    barrage_rate = np.zeros(len(centers))
    window_size = 30.0  # seconds
    
    # Extract barrage epoch data
    if hasattr(barrage_epochs, 'data') and len(barrage_epochs.data) > 0:
        barrage_data = barrage_epochs.data  # shape (n_barrages, 2) with [start, stop]
        
        for i, center in enumerate(centers):
            # Define window around center
            win_start = center - window_size / 2
            win_stop = center + window_size / 2
            
            # Count barrages that overlap with this window
            # A barrage overlaps if: barrage_start < win_stop AND barrage_stop > win_start
            overlaps = (barrage_data[:, 0] < win_stop) & (barrage_data[:, 1] > win_start)
            n_barrages = np.sum(overlaps)
            
            # Convert to rate (events per second)
            barrage_rate[i] = n_barrages / window_size
        
        ca1_complete_df["barrage_rate"] = barrage_rate
        print(f"Added 'barrage_rate' column (mean: {np.mean(barrage_rate):.4f} Hz)")
        print(f"Barrage rate range: {np.min(barrage_rate):.4f} to {np.max(barrage_rate):.4f} Hz")
        print(f"Total barrages detected: {len(barrage_data)}")
    else:
        print("No barrage epochs found - setting barrage_rate to 0")
        ca1_complete_df["barrage_rate"] = 0.0
        
except Exception as e:
    # General exception for unexpected errors
    print(f"No barrage file: {e}")
    barrage_epochs = []
    ca1_complete_df["barrage_rate"] = 0.0
    print("Set barrage_rate to 0.0")

# Compute ripple rate for each center_s with 30s window
print("" + "="*50)
print("Computing ripple rate with 30s window:")
print("="*50)

centers = ca1_complete_df["center_s"].values
ripple_rate = np.zeros(len(centers))
window_size = 30.0  # seconds

# Extract ripple epoch data
if hasattr(ripple_epochs, 'data') and len(ripple_epochs.data) > 0:
    ripple_data = ripple_epochs.data  # shape (n_ripples, 2) with [start, stop]
    
    for i, center in enumerate(centers):
        # Define window around center
        win_start = center - window_size / 2
        win_stop = center + window_size / 2
        
        # Count ripples that overlap with this window
        # A ripple overlaps if: ripple_start < win_stop AND ripple_stop > win_start
        overlaps = (ripple_data[:, 0] < win_stop) & (ripple_data[:, 1] > win_start)
        n_ripples = np.sum(overlaps)
        
        # Convert to rate (events per second)
        ripple_rate[i] = n_ripples / window_size
    
    ca1_complete_df["ripple_rate"] = ripple_rate
    print(f"Added 'ripple_rate' column (mean: {np.mean(ripple_rate):.4f} Hz)")
    print(f"Ripple rate range: {np.min(ripple_rate):.4f} to {np.max(ripple_rate):.4f} Hz")
    print(f"Total ripples detected: {len(ripple_data)}")
else:
    print("No ripple epochs found - setting ripple_rate to 0")
    ca1_complete_df["ripple_rate"] = 0.0


# Compute ripple rate for each center_s with 30s window
print("" + "="*50)
print("Computing ripple rate with 30s window:")
print("="*50)

centers = ca1_complete_df["center_s"].values
ripple_rate = np.zeros(len(centers))
window_size = 30.0  # seconds

# Extract ripple epoch data
if hasattr(ripple_epochs, 'data') and len(ripple_epochs.data) > 0:
    ripple_data = ripple_epochs.data  # shape (n_ripples, 2) with [start, stop]
    
    for i, center in enumerate(centers):
        # Define window around center
        win_start = center - window_size / 2
        win_stop = center + window_size / 2
        
        # Count ripples that overlap with this window
        # A ripple overlaps if: ripple_start < win_stop AND ripple_stop > win_start
        overlaps = (ripple_data[:, 0] < win_stop) & (ripple_data[:, 1] > win_start)
        n_ripples = np.sum(overlaps)
        
        # Convert to rate (events per second)
        ripple_rate[i] = n_ripples / window_size
    
    ca1_complete_df["ripple_rate"] = ripple_rate
    print(f"Added 'ripple_rate' column (mean: {np.mean(ripple_rate):.4f} Hz)")
    print(f"Ripple rate range: {np.min(ripple_rate):.4f} to {np.max(ripple_rate):.4f} Hz")
    print(f"Total ripples detected: {len(ripple_data)}")
else:
    print("No ripple epochs found - setting ripple_rate to 0")
    ca1_complete_df["ripple_rate"] = 0.0



Computing barrage rate with 30s window:
Added 'barrage_rate' column (mean: 0.0033 Hz)
Barrage rate range: 0.0000 to 0.1333 Hz
Total barrages detected: 56
Computing ripple rate with 30s window:
Added 'ripple_rate' column (mean: 0.2103 Hz)
Ripple rate range: 0.0000 to 1.1667 Hz
Total ripples detected: 3004
Computing ripple rate with 30s window:
Added 'ripple_rate' column (mean: 0.2103 Hz)
Ripple rate range: 0.0000 to 1.1667 Hz
Total ripples detected: 3004


In [196]:
# Save ca1_complete_df as pickle file
import pickle
import os

# Define save directory
save_dir = r"D:\Documents\GitHub\Sleep_criticaltiy\data\rats\barr_CA2_CA3"

# Create directory if it doesn't exist
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
    print(f"Created directory: {save_dir}")

# Create filename from animal_name and day
filename = f"{animal_name}_{day}.pkl"
save_path = os.path.join(save_dir, filename)

# Save dataframe
with open(save_path, 'wb') as f:
    pickle.dump(ca1_complete_df, f)

print(f"\n✅ Saved ca1_complete_df to: {save_path}")
print(f"📊 Shape: {ca1_complete_df.shape}")
print(f"📊 Columns: {list(ca1_complete_df.columns)}")
print(f"📊 File size: {os.path.getsize(save_path) / 1024:.2f} KB")


✅ Saved ca1_complete_df to: D:\Documents\GitHub\Sleep_criticaltiy\data\rats\barr_CA2_CA3\AYA10_day34.pkl
📊 Shape: (5825, 15)
📊 Columns: ['center_s', 'd2', 'd2_err', 'state', 'CA1_All_FR_per_unit', 'CA1_E_FR_per_unit', 'CA1_I_FR_per_unit', 'CA2_All_FR_per_unit', 'CA2_E_FR_per_unit', 'CA2_I_FR_per_unit', 'CA3_All_FR_per_unit', 'CA3_E_FR_per_unit', 'CA3_I_FR_per_unit', 'barrage_rate', 'ripple_rate']
📊 File size: 678.38 KB
